# grilly2 on Colab

The torch API on Vulkan, on a Colab NVIDIA GPU.

**Runtime → Change runtime type → T4 GPU**, then **Runtime → Run all**.

1. **Register Colab's NVIDIA driver with Vulkan.** This is the one Colab-specific step.
2. **Install the LunarG Vulkan SDK**, for `vulkaninfo` and for a source build.
3. **Install grilly2**, from PyPI or from source.
4. **Check the device and compare against torch on CUDA.**
5. **Generate text** with Qwen2.5-0.5B, quantized to int8.
6. **Run the test suite on the GPU** (optional).

grilly2 is for the consumer AMD and Intel cards that torch will not train on.
On Colab it runs on NVIDIA through the same Vulkan path, which makes Colab a free
way to try it on real GPU hardware. On a CPU-only runtime the notebook falls back to
Mesa's lavapipe, a software Vulkan driver: correct, but slow.

In [ ]:
SDK_VERSION = "1.4.363.1"   # LunarG Vulkan SDK; the newest is at https://vulkan.lunarg.com/sdk/latest/linux.txt
BUILD_FROM_SOURCE = False   # True: build grilly2 from GRILLY_REF against the SDK (1.2 min on a T4 runtime)
GRILLY_REF = "main"         # branch, tag or commit for the source build and the test suite
RUN_TESTS = False           # True: run the Python test suite on this GPU, and the Rust one if building from source

## 1. Vulkan on Colab's NVIDIA GPU

Colab mounts the NVIDIA driver's user-space libraries in `/usr/lib64-nvidia`, the
Vulkan driver among them, but does not register that driver with anything. These
results are from a T4 with driver 580.82.07:

- `/etc/vulkan/icd.d/` is empty, so the Vulkan loader finds no driver at all
  (`VK_ERROR_INCOMPATIBLE_DRIVER`).
- With a driver manifest alone, it still fails: the driver returns NULL for
  `vkCreateInstance`. NVIDIA's Vulkan driver starts up through glvnd's EGL, and only
  Mesa's EGL vendor is registered.

Two JSON files fix both. Nothing is installed. An apt `libnvidia-gl-*` package would add a
second copy of the driver, whose version need not match the kernel module's.

In [ ]:
import glob, json, os, shutil, subprocess, time

def sh(cmd):
    """Run a shell command, show its output as it comes, and stop here if it fails."""
    print("$", cmd, flush=True)
    proc = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in proc.stdout:
        print(line, end="", flush=True)
    if proc.wait():
        raise RuntimeError(f"exit status {proc.returncode}: {cmd}")

def write_json(path, value):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w") as f:
        json.dump(value, f, indent=2)
    print("wrote", path, "->", value["ICD"]["library_path"])

def find_lib(name):
    """`name` in the first directory of the library path that holds it."""
    dirs = os.environ.get("LD_LIBRARY_PATH", "").split(":")
    dirs += ["/usr/lib64-nvidia", "/usr/lib/x86_64-linux-gnu", "/lib/x86_64-linux-gnu"]
    for d in filter(None, dirs):
        if os.path.exists(os.path.join(d, name)):
            return os.path.join(d, name)

gpu = None
if shutil.which("nvidia-smi"):
    gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,driver_version,memory.total",
                          "--format=csv,noheader"], capture_output=True, text=True).stdout.strip()

if gpu:
    print("GPU:", gpu)
    glx, egl = find_lib("libGLX_nvidia.so.0"), find_lib("libEGL_nvidia.so.0")
    if not (glx and egl):
        raise RuntimeError(
            f"This runtime does not mount the NVIDIA driver's Vulkan library (libGLX_nvidia.so.0: {glx}, "
            f"libEGL_nvidia.so.0: {egl}). Colab has changed how it provides the driver; please open an "
            "issue at https://github.com/Grillcheese-AI/grilly2/issues with this message.")
    # the Vulkan driver manifest, unless this image already ships one
    present = glob.glob("/etc/vulkan/icd.d/*nvidia*.json") + glob.glob("/usr/share/vulkan/icd.d/*nvidia*.json")
    if present:
        print("already registered:", present)
    else:
        write_json("/etc/vulkan/icd.d/nvidia_icd.json",
                   {"file_format_version": "1.0.1", "ICD": {"library_path": glx, "api_version": "1.3.0"}})
    # the EGL vendor the Vulkan driver initialises through
    vendor = "/usr/share/glvnd/egl_vendor.d/10_nvidia.json"
    if os.path.exists(vendor):
        print("already registered:", vendor)
    else:
        write_json(vendor, {"file_format_version": "1.0.0", "ICD": {"library_path": egl}})
else:
    print("No NVIDIA GPU in this runtime: using Mesa's lavapipe, a software Vulkan driver (slow).")
    sh("apt-get -qq update && apt-get -qq install -y mesa-vulkan-drivers > /dev/null")
    os.environ["ALLOW_CPU_VULKAN"] = "1"   # grilly refuses a CPU device unless asked

if "libvulkan.so.1" not in subprocess.run(["ldconfig", "-p"], capture_output=True, text=True).stdout:
    sh("apt-get -qq update && apt-get -qq install -y libvulkan1 > /dev/null")

## 2. The Vulkan SDK

The grilly2 wheel does not need the SDK; running it takes only the driver. The SDK is
here for three things:

- `vulkaninfo`;
- a source build, which needs the headers, `glslc` and shaderc;
- its Vulkan loader, which is newer than Ubuntu's. Every program started from this
  notebook uses that loader. The Python kernel keeps the system loader it started
  with; the check in step 4 prints which one it has.

In [ ]:
SDK = f"/content/vulkan/{SDK_VERSION}/x86_64"
if not os.path.isdir(SDK):
    url = f"https://sdk.lunarg.com/sdk/download/{SDK_VERSION}/linux/vulkansdk-linux-x86_64-{SDK_VERSION}.tar.xz"
    sh(f"mkdir -p /content/vulkan && curl -fsSL {url} | tar -xJ -C /content/vulkan")
os.environ["VULKAN_SDK"] = SDK
os.environ["PATH"] = f"{SDK}/bin:{os.environ['PATH']}"
# 1.4.363 keeps the loader in lib/VulkanLoader/lib rather than lib/; LIBRARY_PATH lets
# the linker find it there, so a branch that predates that move still builds
loader_dir = f"{SDK}/lib/VulkanLoader/lib"
os.environ["LD_LIBRARY_PATH"] = ":".join(filter(None, [os.environ.get("LD_LIBRARY_PATH"), loader_dir, f"{SDK}/lib"]))
os.environ["LIBRARY_PATH"] = ":".join(filter(None, [loader_dir, os.environ.get("LIBRARY_PATH")]))
sh("vulkaninfo --summary 2>/dev/null | sed -n '/^Vulkan Instance Version/p;/^Devices:/,$p'")

## 3. Install grilly2

On PyPI the package is `grilly2`, and you import it as `grilly`. The source build is
the one a contributor runs: Rust and maturin over this checkout, against the SDK from
step 2.

In [ ]:
REPO = "/content/grilly2"

def checkout():
    if not os.path.isdir(REPO):
        sh(f"git clone -q https://github.com/Grillcheese-AI/grilly2 {REPO}")
    sh(f"cd {REPO} && git fetch -q origin {GRILLY_REF} && git checkout -q FETCH_HEAD "
       "&& git submodule update -q --init --recursive && git log -1 --format='%h %s'")

if BUILD_FROM_SOURCE:
    cargo_bin = os.path.expanduser("~/.cargo/bin")
    if not os.path.exists(f"{cargo_bin}/cargo"):
        sh("curl -fsSL https://sh.rustup.rs | sh -s -- -y -q --profile minimal")
    os.environ["PATH"] = f"{cargo_bin}:{os.environ['PATH']}"
    checkout()
    sh("pip -q install maturin")
    start = time.perf_counter()
    sh(f"cd {REPO} && rm -rf dist && maturin build --release --out dist --auditwheel skip")
    print(f"built in {(time.perf_counter() - start) / 60:.1f} min")
    wheel = glob.glob(f"{REPO}/dist/*.whl")[0]
    sh(f"pip -q uninstall -y grilly2; pip -q install '{wheel}[huggingface]'")
else:
    sh("pip -q install 'grilly2[huggingface]'")

## 4. Check it

`grilly-doctor` reports the interpreter, the install and the device, then runs a real
kernel. After that comes a small forward and backward pass, compared against torch
running on the same GPU through CUDA.

In [ ]:
sh("grilly-doctor")

import numpy as np
import torch as pt   # Colab's torch, on CUDA
import grilly        # the same API, on Vulkan

grilly.manual_seed(0)
x = grilly.randn(256, 128, requires_grad=True)
lin = grilly.nn.Linear(128, 64)
loss = (grilly.nn.functional.gelu(lin(x)) ** 2).mean()
loss.backward()

dev = "cuda" if pt.cuda.is_available() else "cpu"
X = pt.tensor(x.detach().numpy(), device=dev, requires_grad=True)
W = pt.tensor(lin.weight.detach().numpy(), device=dev)
b = pt.tensor(lin.bias.detach().numpy(), device=dev)
ref = (pt.nn.functional.gelu(X @ W.T + b) ** 2).mean()
ref.backward()

print(f"loss          grilly {loss.item():.7f}   torch ({dev}) {ref.item():.7f}")
print(f"max |dx| diff {np.abs(x.grad.numpy() - X.grad.cpu().numpy()).max():.2e}")
maps = open("/proc/self/maps").read().split("\n")
print("this kernel's Vulkan loader:", next((l.split()[-1] for l in maps if "libvulkan" in l), "not loaded"))

## 5. Generate text

This is grilly's inference acceptance run:

1. load a small open-weights model from its safetensors checkpoint;
2. quantize it to int8 as it loads;
3. sample on the GPU through `generate()`, which takes Hugging Face's arguments.

`grilly.huggingface` is a thin adapter over `transformers`' API. The machinery under
it (`grilly.infer`) works without the adapter installed.

In [ ]:
import time
from importlib.metadata import version
from grilly.huggingface import AutoModelForCausalLM, AutoTokenizer
from grilly.infer.quantization import Int8WeightOnlyConfig

# grilly2 0.1.0's attention kernels need 53 KiB of shared memory per workgroup and NVIDIA
# allows 48: on an NVIDIA GPU its first prefill loses the device and takes the runtime down
if gpu and not BUILD_FROM_SOURCE and version("grilly2") == "0.1.0":
    raise SystemExit("grilly2 0.1.0 cannot run attention on NVIDIA GPUs (fixed after 0.1.0). "
                     "Set BUILD_FROM_SOURCE = True in the first cell, or install a newer grilly2.")

MODEL = "Qwen/Qwen2.5-0.5B-Instruct"
model = AutoModelForCausalLM.from_pretrained(MODEL, quantization_config=Int8WeightOnlyConfig())
tokenizer = AutoTokenizer.from_pretrained(MODEL)
ids = tokenizer.apply_chat_template([{"role": "user", "content": "Why is the sky blue? Answer in three sentences."}],
                                    add_generation_prompt=True, return_tensors="pt")

grilly.manual_seed(0)
start = time.perf_counter()
out = model.generate(ids, max_new_tokens=128, do_sample=True, top_p=0.8)
elapsed = time.perf_counter() - start

new = out[0].tolist()[ids.shape[-1]:]
print(tokenizer.decode(new))
print(f"\n{len(new)} tokens in {elapsed:.2f} s, {len(new) / elapsed:.1f} tokens/s including the prompt")
used = subprocess.run(["nvidia-smi", "--query-gpu=memory.used", "--format=csv,noheader"],
                      capture_output=True, text=True).stdout.strip() if gpu else "n/a"
print(f"GPU memory in use: {used}")

## 6. The test suite on this GPU

CI has no GPU, so it runs these suites on lavapipe. There it skips, by name, the tests
that assert a precision real GPUs have and Vulkan does not promise. Here nothing is
skipped: a failure on this GPU is a finding worth an issue.

The torch and transformers references are pinned to the versions CI uses. That
replaces Colab's torch in this runtime and removes its torchvision and torchaudio,
which are built for the newer torch and break `transformers`' imports. Run this
step last.

In [ ]:
if RUN_TESTS:
    checkout()
    sh("pip -q uninstall -y torchvision torchaudio")   # built for Colab's torch, not the pinned one
    sh("pip -q install torch==2.10.0 --index-url https://download.pytorch.org/whl/cpu")
    sh("pip -q install pytest pytest-timeout transformers==5.5.1")
    # from outside the checkout, so `import grilly` is the installed package
    sh(f"cd /content && python -m pytest {REPO}/tests/python --rootdir {REPO} -p no:cacheprovider --timeout 600 -ra -q")
    if BUILD_FROM_SOURCE:
        sh(f"cd {REPO} && GRILLY_PYTHON=python cargo test --workspace --release --no-fail-fast -- --test-threads=2")
else:
    print("Set RUN_TESTS = True in the first cell to run the suites here.")